# Housing cost burden - pull, clean, export

Indicator: percent of renter-occupied units paying 35% or more of household income on gross rent.
Census ACS5 Data Profile DP04, variable DP04_0142PE.

Note the dataset path below is `acs/acs5/profile`, not `acs/acs5/subject` - a different ACS
product (Data Profiles vs. Subject Tables). Easy to copy-paste wrong from the other notebooks.

Same `CENSUS_API_KEY` as the other Census pulls.

In [ ]:
import json
import os
import time
from datetime import date
from pathlib import Path

import pandas as pd
import requests

API_KEY = os.environ.get("CENSUS_API_KEY")
if not API_KEY:
    raise RuntimeError("CENSUS_API_KEY not set. Set it as an environment variable, do not hardcode it here.")

RAW_DIR = Path("../raw")
DATA_DIR = Path("../data")
RAW_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

today = date.today().isoformat()
print("Pipeline run date:", today)

In [ ]:
VARIABLE = "DP04_0142PE"  # Percent of renter-occupied units at 35%+ GRAPI
DATASET = "acs/acs5/profile"  # Data Profile, not Subject Table
YEARS = [2014, 2016, 2018, 2020, 2022]

GEOGRAPHIES = {
    "Allegheny County": {"for": "county:003", "in": "state:42"},
    "Pennsylvania": {"for": "state:42"},
    "United States": {"for": "us:1"},
}

In [ ]:
rows = []
raw_responses = {}

for year in YEARS:
    url = f"https://api.census.gov/data/{year}/{DATASET}"
    for geo_name, geo_params in GEOGRAPHIES.items():
        params = {"get": f"NAME,{VARIABLE}", "key": API_KEY, **geo_params}
        try:
            resp = requests.get(url, params=params, timeout=30)
            resp.raise_for_status()
            payload = resp.json()
        except Exception as exc:
            print(f"FAILED  year={year}  geo={geo_name}  error={exc}")
            continue

        raw_responses[f"{year}_{geo_name}"] = payload

        header, data_row = payload[0], payload[1]
        record = dict(zip(header, data_row))
        value_raw = record.get(VARIABLE)

        value = None
        if value_raw is not None:
            try:
                value_f = float(value_raw)
                value = None if value_f < 0 else value_f
            except ValueError:
                value = None

        rows.append({"year": year, "geography": geo_name, "value": value})
        time.sleep(0.2)

with open(RAW_DIR / f"housing_cost_burden_raw_{today}.json", "w") as f:
    json.dump(raw_responses, f, indent=2)

df = pd.DataFrame(rows)
df

In [ ]:
expected = len(YEARS) * len(GEOGRAPHIES)
print(f"Expected {expected} rows, got {len(df)}")
print(df[df["value"].isna()])

In [ ]:
wide = df.pivot(index="year", columns="geography", values="value").reset_index()
series = wide.to_dict(orient="records")

payload = {
    "indicator": "Renter households paying 35%+ of income on gross rent (%)",
    "geographies": list(GEOGRAPHIES.keys()),
    "unit": "percent",
    "snapshot_as_of": today,
    "status": "live",
    "source": f"U.S. Census Bureau, American Community Survey 5-Year Estimates, Data Profile DP04, variable {VARIABLE}",
    "series": series,
    "notes": "",
}

with open(DATA_DIR / "housing_cost_burden.json", "w") as f:
    json.dump(payload, f, indent=2)

print("Wrote", DATA_DIR / "housing_cost_burden.json")
payload